# 내시경 폴립 분할 (Kvasir-SEG) 불균형 세그멘테이션 실험

---

## 1. 태스크 및 도메인
- **도메인**: 내시경 폴립 (Endoscopic Polyp) 세그멘테이션
- **모달리티**: Endoscopy (내시경 RGB 이미지)
- **태스크**: Binary segmentation — 배경(0) / 폴립(1)
- **핵심 도전**: 폴립 색상·형태·크기의 다양성, 조명 반사, 점막 패턴과의 유사성

## 2. 모델
- **아키텍처**: PraNet (Res2Net50 백본, 역방향 어텐션)
- **사전학습**: ImageNet pretrained (Res2Net50)
- **선택 이유**: 폴립 세그멘테이션 특화 모델, 멀티스케일 특성 추출에 강점
- **출력**: 1채널 sigmoid → `to_2ch_logits()` 변환 후 손실 함수 적용
- **주의**: PraNet 상대 import 패치 필요 (`from .Res2Net_v1b` → `from Res2Net_v1b`)

## 3. 데이터셋
- **이름**: Kvasir-SEG
- **규모**: 1,000장 — Train 800 / Val 200 (8:1:1 랜덤 분할), 입력 352×352
- **클래스 불균형**: BG:Polyp = **~5.4:1** (비교적 완만한 불균형)
- **공식 분할**: 없음 → random_state=42, 8:1:1 분할

## 4. 데이터 준비 (협업자용)
> Cell 0 실행 시 자동으로 데이터가 다운로드됨. 별도 준비 불필요.

**취득 방법 (자동)**:
```python
kagglehub.dataset_download("meetnagadia/kvasir-dataset")
```
공식 사이트: https://datasets.simula.no/kvasir-seg/

## 5. 전처리 및 도메인 특이점
- BGR→RGB 변환, 이미지/마스크 동일 크기 리사이즈 (352×352)
- 마스크 이진화: 임계값 128
- ImageNet 정규화
- 증강: HorizontalFlip, VerticalFlip, RandomRotate90
- PraNet 출력: 다중 사이드 출력(lateral map) → 각 출력에 손실 함수 합산 적용

## 6. 실험 손실 함수 및 Optuna 탐색 범위
| 손실 함수 | 탐색 파라미터 | 탐색 범위 | Trials |
|-----------|-------------|----------|--------|
| `ce_dice` | — | — | — |
| `wce_dice` | — | — | — |
| `lwce_dice` | — | — | — |
| `plwce_dice` | alpha | 2.5 ~ 15.0 | 20 |
| `pwce_dice` | alpha | 0.2 ~ 2.5 | 20 |
| `cb_dice` | — | — | — |
| `plwce_focal_dice` | alpha + gamma | alpha 2.5~15.0, gamma 0.5~5.0 | 40 |

## 7. SoTA 참고 (2026년 3월 기준)
| 방법 | Dice | IoU | 출처 |
|------|------|-----|------|
| MNet-SAt (2024) | **96.61%** | — | IEEE Access |
| ARCUNet (2025) | 95.34% | **93.53%** | arXiv |
| PraNet (2021, 원본) | 89.8% | 84.0% | MICCAI'21 |
| U-Net baseline | ~79~82% | — | 복수 논문 |

> 본 연구 목표: PraNet baseline 대비 LWCE/PLWCE 계열 손실 함수의 개선 효과 검증.
> 평가 지표: Dice, Sensitivity, Specificity, AUC
> 결과 저장: `medical_data/results/Endoscopic_Polyp_Image/`

In [ ]:
# === Cell 0: 환경 설정 ===
import os, sys, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import cv2
import glob
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# custom_losses.py (로컬 경로)
sys.path.insert(0, '/root/imbalanced-data-LWCE/medical_data')
from custom_losses import get_loss_function

# PraNet 설정 (로컬 /tmp/PraNet)
pranet_lib = '/tmp/PraNet/lib'
if pranet_lib not in sys.path:
    sys.path.insert(0, pranet_lib)

if not os.path.exists('/tmp/PraNet'):
    os.system('git clone https://github.com/DengPingFan/PraNet.git /tmp/PraNet')
    target = '/tmp/PraNet/lib/PraNet_Res2Net.py'
    with open(target) as f: code = f.read()
    if 'from .Res2Net_v1b' in code:
        with open(target, 'w') as f: f.write(code.replace('from .Res2Net_v1b', 'from Res2Net_v1b'))

wp = '/tmp/PraNet/models/res2net50_v1b_26w_4s-3cf99910.pth'
os.makedirs('/tmp/PraNet/models', exist_ok=True)
if not os.path.exists(wp):
    print("Res2Net 가중치 다운로드 중...")
    urllib.request.urlretrieve(
        'https://shanghuagao.oss-cn-beijing.aliyuncs.com/res2net/res2net50_v1b_26w_4s-3cf99910.pth', wp)

r2n = '/tmp/PraNet/lib/Res2Net_v1b.py'
with open(r2n) as f: code = f.read()
hc = '/media/nercms/NERCMS/GepengJi/Medical_Seqmentation/CRANet/models/res2net50_v1b_26w_4s-3cf99910.pth'
if hc in code:
    with open(r2n, 'w') as f: f.write(code.replace(hc, wp))

for k in [k for k in sys.modules if 'Res2Net' in k or 'PraNet_Res2Net' in k]:
    del sys.modules[k]
from PraNet_Res2Net import PraNet

# --- 결과 저장 경로 ---
RESULTS_DIR = '/root/imbalanced-data-LWCE/medical_data/results/Endoscopic_Polyp_Image'
os.makedirs(RESULTS_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print("PraNet 로드 완료")


In [ ]:
# === Cell 1: 데이터 로드 ===
# Kvasir-SEG 데이터셋 (kagglehub 캐시 활용)
DATA_DIR = '/root/.cache/kagglehub/datasets/debeshjha1/kvasirseg/versions/3/Kvasir-SEG/Kvasir-SEG'
if not os.path.exists(DATA_DIR):
    import kagglehub
    path = kagglehub.dataset_download('debeshjha1/kvasirseg')
    DATA_DIR = os.path.join(path, 'Kvasir-SEG', 'Kvasir-SEG')

class PolypDataset(Dataset):
    def __init__(self, imgs, masks, transform=None):
        self.imgs    = sorted(imgs)
        self.masks   = sorted(masks)
        self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        img  = cv2.cvtColor(cv2.imread(self.imgs[i]),  cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.masks[i], cv2.IMREAD_GRAYSCALE)
        _, mask = cv2.threshold(mask, 127, 1, cv2.THRESH_BINARY)
        if self.transform:
            aug  = self.transform(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']
        return img, mask.long()

all_images = sorted(glob.glob(os.path.join(DATA_DIR, 'images', '*.jpg')))
all_masks  = sorted(glob.glob(os.path.join(DATA_DIR, 'masks',  '*.jpg')))

# 80/10/10 분할
tr_imgs, tmp_imgs, tr_masks, tmp_masks = train_test_split(
    all_images, all_masks, test_size=0.2, random_state=42)
val_imgs, test_imgs, val_masks, test_masks = train_test_split(
    tmp_imgs, tmp_masks, test_size=0.5, random_state=42)

train_tf = A.Compose([A.Resize(352,352), A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
                       A.RandomRotate90(p=0.5), A.Normalize(), ToTensorV2()])
val_tf   = A.Compose([A.Resize(352,352), A.Normalize(), ToTensorV2()])

train_loader = DataLoader(PolypDataset(tr_imgs,  tr_masks,  train_tf), batch_size=8, shuffle=True,  num_workers=2)
val_loader   = DataLoader(PolypDataset(val_imgs, val_masks, val_tf),   batch_size=8, shuffle=False, num_workers=2)
test_loader  = DataLoader(PolypDataset(test_imgs, test_masks, val_tf), batch_size=8, shuffle=False, num_workers=2)

# 클래스 비율 계산
print("클래스 비율 계산 중...")
bg, fg = 0, 0
for mp in tr_masks:
    m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
    fg += int((m > 127).sum()); bg += int((m <= 127).sum())
class_counts = [bg, fg]
print(f"Train {len(tr_imgs)} | Val {len(val_imgs)} | Test {len(test_imgs)}")
print(f"BG: {bg:,}  FG(polyp): {fg:,}  Ratio: {bg/fg:.1f}:1")


In [ ]:
# === Cell 2: 모델 정의 ===
# 1채널 → 2채널 대칭 logit (핵심 버그 수정)
def to_2ch_logits(p):
    return torch.cat([-p, p], dim=1)

def compute_val_dice(model, loader):
    model.eval()
    ds = 0.0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            _, _, _, res = model(imgs)
            res  = F.interpolate(res, size=masks.shape[1:], mode='bilinear', align_corners=True)
            prob = torch.sigmoid(res).squeeze(1)
            pred = (prob > 0.5).long()
            inter = (pred.float() * masks.float()).sum()
            union = pred.float().sum() + masks.float().sum()
            ds += (2. * inter / (union + 1e-8)).item() if union > 0 else 1.0
    return ds / len(loader)

print("유틸리티 함수 정의 완료")


In [ ]:
# === Cell 3: 학습 함수 ===
def train_pranet(loss_name, alpha=1.0, gamma=2.0, epochs=20, lr=1e-4):
    model     = PraNet().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = get_loss_function(loss_name, class_counts=class_counts, alpha=alpha, gamma=gamma)
    print(f"\n{'='*50}\nPraNet + {loss_name}  (Epochs={epochs})\n{'='*50}")

    history   = {'loss': [], 'val_dice': []}
    best_dice = 0.0

    for epoch in range(epochs):
        model.train(); tl = 0.0
        for imgs, masks in tqdm(train_loader, desc=f"Ep{epoch+1:02d}/{epochs}", leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            o5, o4, o3, o2 = model(imgs)
            loss = 0
            for out in [o5, o4, o3, o2]:
                out  = F.interpolate(out, size=masks.shape[1:], mode='bilinear', align_corners=True)
                loss += criterion(to_2ch_logits(out), masks)
            loss.backward(); optimizer.step(); tl += loss.item()

        avg_l = tl / len(train_loader)
        vdice = compute_val_dice(model, val_loader)
        history['loss'].append(avg_l)
        history['val_dice'].append(vdice)
        print(f"Ep{epoch+1:02d} | Loss:{avg_l:.4f} | ValDice:{vdice:.4f}", end="")
        if vdice > best_dice:
            best_dice = vdice
            torch.save(model.state_dict(), f'/tmp/best_pranet_{loss_name}_a{alpha:.2f}_g{gamma:.2f}.pth')
            print("  ← Best!", end="")
        print()

    model.load_state_dict(torch.load(f'/tmp/best_pranet_{loss_name}_a{alpha:.2f}_g{gamma:.2f}.pth', weights_only=True))
    print(f"최고 Val Dice: {best_dice:.4f}")
    return model, history, best_dice

print("train_pranet 함수 준비 완료")


In [ ]:
# === Cell 4: Optuna alpha/gamma 탐색 ===
import os as _os
_os.environ['TQDM_DISABLE'] = '1'
import optuna as _optuna, json as _json
import traceback as _tb
_optuna.logging.set_verbosity(_optuna.logging.WARNING)

ALPHA_LOW_PF  = 2.5;  ALPHA_HIGH_PF = 15.0
GAMMA_LOW_PF  = 0.5;  GAMMA_HIGH_PF = 5.0
N_TRIALS_PF   = 60
PROXY_EPOCHS_PF = 8

def objective_pf(trial):
    alpha = trial.suggest_float('alpha', ALPHA_LOW_PF, ALPHA_HIGH_PF)
    gamma = trial.suggest_float('gamma', GAMMA_LOW_PF, GAMMA_HIGH_PF)
    try:
        _, _, dice = train_pranet('plwce_focal_dice', alpha=alpha, gamma=gamma, epochs=PROXY_EPOCHS_PF)
        return dice
    except Exception as e:
        print(f"Trial {trial.number} 실패: {e}")
        _tb.print_exc()
        return 0.0

N_ALPHA_GRID = 10  # 10x6=60 grid
N_GAMMA_GRID = 6
sampler_pf = _optuna.samplers.GridSampler({
    'alpha': np.linspace(ALPHA_LOW_PF, ALPHA_HIGH_PF, N_ALPHA_GRID).tolist(),
    'gamma': np.linspace(GAMMA_LOW_PF, GAMMA_HIGH_PF, N_GAMMA_GRID).tolist(),
})
study_pf = _optuna.create_study(
    direction='maximize', study_name='kvasir_plwce_focal',
    sampler=sampler_pf
)
study_pf.optimize(objective_pf, n_trials=N_TRIALS_PF, show_progress_bar=True)

best_alpha_pf = study_pf.best_params['alpha']
best_gamma_pf = study_pf.best_params['gamma']
print(f"\n[PLWCE+Focal] 최적 alpha={best_alpha_pf:.4f}, gamma={best_gamma_pf:.4f}  (Val Dice={study_pf.best_value:.4f})")

# 탐색 결과 시각화 (2D scatter)
trials_pf = [t for t in study_pf.trials if t.value is not None]
alphas_pf  = [t.params['alpha'] for t in trials_pf]
gammas_pf  = [t.params['gamma'] for t in trials_pf]
values_pf  = [t.value           for t in trials_pf]

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(alphas_pf, gammas_pf, c=values_pf, cmap='viridis', s=60, alpha=0.8)
ax.scatter([best_alpha_pf], [best_gamma_pf], color='red', s=150, marker='*', zorder=5,
           label=f'Best α={best_alpha_pf:.2f}, γ={best_gamma_pf:.2f}')
plt.colorbar(sc, ax=ax, label='Val Dice')
ax.set_xlabel('alpha'); ax.set_ylabel('gamma')
ax.set_title('PLWCE+Focal Optuna 탐색 (Kvasir)'); ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'Kvasir_optuna_search_pf.png'), dpi=100)
plt.show()

# Optuna 결과 저장
optuna_save = {
    'plwce_focal': {
        'best_alpha': best_alpha_pf, 'best_gamma': best_gamma_pf,
        'best_proxy_dice': study_pf.best_value,
        'trials': [{'number': t.number, 'alpha': t.params.get('alpha'),
                    'gamma': t.params.get('gamma'), 'value': t.value}
                   for t in study_pf.trials if t.value is not None]
    }
}
with open(os.path.join(RESULTS_DIR, 'Kvasir_optuna_results_pf.json'), 'w') as f:
    _json.dump(optuna_save, f, indent=2, ensure_ascii=False)
print(f"Optuna 결과 저장: {os.path.join(RESULTS_DIR, 'Kvasir_optuna_results_pf.json')}")


In [ ]:
DOMAIN = 'Kvasir'
# === Cell 5: 전체 Loss 비교 학습 ===
LOSS_LIST   = ['ce_dice', 'wce_dice', 'lwce_dice', 'plwce_dice', 'cb_dice', 'plwce_focal_dice']
all_results = {}

for loss_name in LOSS_LIST:
    if loss_name == 'plwce_focal_dice':
        m, h, b = train_pranet(loss_name, alpha=best_alpha_pf, gamma=best_gamma_pf, epochs=20)
    else:
        m, h, b = train_pranet(loss_name, epochs=20)
    all_results[loss_name] = {'model': m, 'history': h, 'best_dice': b}

print("\n[Loss 비교 실험 결과 요약]")
print(f"{'Loss':<15} {'Best Val Dice':>13}")
print("-" * 30)
for k, v in all_results.items():
    print(f"{k:<15} {v['best_dice']:>13.4f}")

# --- 학습 곡선 시각화 ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for k, v in all_results.items():
    h = v['history']
    ax1.plot(h['loss'],     label=k)
    ax2.plot(h['val_dice'], label=k)
ax1.set_title('Train Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True)
ax2.set_title('Val Dice');   ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_training_curves.png'), dpi=100)
plt.show()
print(f"학습 곡선 저장: {os.path.join(RESULTS_DIR, f'{DOMAIN}_training_curves.png')}")


In [ ]:
# === Cell 6: 평가 및 결과 저장 ===
import json, pandas as pd
from sklearn.metrics import confusion_matrix as _cm

def predict_polyp(model, img_path):
    model.eval()
    img    = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h0, w0 = img.shape[:2]
    tf     = A.Compose([A.Resize(352, 352), A.Normalize(), ToTensorV2()])
    tensor = tf(image=img)['image'].unsqueeze(0).to(device)
    with torch.no_grad():
        _, _, _, res = model(tensor)
        prob = torch.sigmoid(res).squeeze().cpu().numpy()
    return cv2.resize(prob, (w0, h0)), img

best_key   = max(all_results, key=lambda k: all_results[k]['best_dice'])
best_model = all_results[best_key]['model']
print(f"최고 모델: {best_key}  (ValDice={all_results[best_key]['best_dice']:.4f})")

# --- 최종 Test Set 정량 평가 ---
from sklearn.metrics import roc_auc_score as _roc_auc

def compute_test_metrics(model, loader):
    model.eval()
    all_dice, all_sens, all_spec, all_auc = [], [], [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            _, _, _, res = model(imgs)
            prob = torch.sigmoid(res).squeeze(1).cpu().numpy()
            pred = (prob > 0.5).astype(int)
            gt   = masks.cpu().numpy()
            for p, g, pr in zip(pred, gt, prob):
                inter = (p & g).sum()
                dice  = 2*inter / (p.sum() + g.sum() + 1e-8)
                all_dice.append(float(dice))
                tn, fp, fn, tp = _cm(g.ravel(), p.ravel(), labels=[0,1]).ravel()
                all_sens.append(float(tp / (tp + fn + 1e-8)))
                all_spec.append(float(tn / (tn + fp + 1e-8)))
                if g.sum() > 0 and g.sum() < g.size:
                    try: all_auc.append(_roc_auc(g.ravel(), pr.ravel()))
                    except: pass
    return {
        'Dice':        float(np.mean(all_dice)),
        'Sensitivity': float(np.mean(all_sens)),
        'Specificity': float(np.mean(all_spec)),
        'AUC':         float(np.mean(all_auc)) if all_auc else 0.0,
    }

print('\n[최종 평가 — Test Set]')
print(f"{'Loss':<20} {'Dice':>7} {'Sens':>7} {'Spec':>7} {'AUC':>7}")
print('-' * 50)
final_results = {}
for k, v in all_results.items():
    m = compute_test_metrics(v['model'], test_loader)
    alpha_val = best_alpha_pf if k == 'plwce_focal_dice' else 1.0
    final_results[k] = {
        'loss_name':    k,
        'alpha':        float(alpha_val),
        'best_val_dice': float(v['best_dice']),
        'Dice':         m['Dice'],
        'Sensitivity':  m['Sensitivity'],
        'Specificity':  m['Specificity'],
        'AUC':          m['AUC'],
    }
    print(f"{k:<20} {m['Dice']:>7.4f} {m['Sensitivity']:>7.4f} {m['Specificity']:>7.4f} {m['AUC']:>7.4f}")

# --- 예측 시각화 ---
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for i in range(3):
    prob, img_rgb = predict_polyp(best_model, test_imgs[i])
    mask = cv2.imread(test_masks[i], cv2.IMREAD_GRAYSCALE)
    pred = (prob > 0.5).astype(np.uint8)
    axes[i,0].imshow(img_rgb);            axes[i,0].set_title('Input');          axes[i,0].axis('off')
    axes[i,1].imshow(mask, cmap='gray');  axes[i,1].set_title('Ground Truth');   axes[i,1].axis('off')
    axes[i,2].imshow(prob,  cmap='jet');  axes[i,2].set_title('Prob Map');       axes[i,2].axis('off')
    axes[i,3].imshow(pred,  cmap='gray'); axes[i,3].set_title(f'Pred({best_key})'); axes[i,3].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_prediction_vis.png'), dpi=100)
plt.show()
print(f"예측 시각화 저장: {RESULTS_DIR}/{DOMAIN}_prediction_vis.png")

# --- 바 차트 (4-metric) ---
COLORS = ['#4878D0','#EE854A','#6ACC65','#D65F5F','#B47CC7','#956CB4']
metrics_to_plot = ['Dice', 'Sensitivity', 'Specificity', 'AUC']
loss_labels = list(final_results.keys())
fig, axes_m = plt.subplots(1, 4, figsize=(18, 5))
for ax, metric in zip(axes_m, metrics_to_plot):
    vals = [final_results[l][metric] for l in loss_labels]
    bars = ax.bar(range(len(loss_labels)), vals,
                  color=[COLORS[i % len(COLORS)] for i in range(len(loss_labels))])
    ax.set_xticks(range(len(loss_labels)))
    ax.set_xticklabels(loss_labels, rotation=30, ha='right', fontsize=8)
    ax.set_title(metric); ax.set_ylabel(metric); ax.set_ylim(0, 1)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', va='bottom', fontsize=7)
plt.suptitle(f'{DOMAIN} — Final Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_final_metrics.png'), dpi=100)
plt.show()
print(f"바차트 저장: {RESULTS_DIR}/{DOMAIN}_final_metrics.png")

# --- JSON 저장 ---
imbalance_ratio = class_counts[0] / class_counts[1]
output_json = {
    'domain':       DOMAIN,
    'model':        'PraNet (Res2Net50)',
    'sota_ref':     {'PraNet (MICCAI 2021)': {'Dice': 0.898}},
    'num_classes':  2,
    'class_counts': class_counts,
    'imbalance':    {'BG_Polyp': round(imbalance_ratio, 2)},
    'train_count':  len(tr_imgs),
    'final_epochs': 20,
    'results':      final_results,
    'best_model':   best_key,
}
json_path = os.path.join(RESULTS_DIR, f'{DOMAIN}_final_results.json')
with open(json_path, 'w') as f:
    json.dump(output_json, f, indent=2, ensure_ascii=False)
print(f'JSON 저장: {json_path}')

# --- Excel 저장 ---
summary_rows = []
for label, m in final_results.items():
    summary_rows.append({
        'Loss_Function':    label,
        'loss_name':        m['loss_name'],
        'alpha':            m['alpha'],
        'Best_Val_Dice':    round(m['best_val_dice'], 4),
        'Test_Dice':        round(m['Dice'],        4),
        'Test_Sensitivity': round(m['Sensitivity'], 4),
        'Test_Specificity': round(m['Specificity'], 4),
        'Test_AUC':         round(m['AUC'],         4),
        'imbalance_ratio':  round(imbalance_ratio,  2),
        'epochs':           20,
        'model':            'PraNet (Res2Net50)',
    })
df_summary = pd.DataFrame(summary_rows)

history_rows = []
for label, v in all_results.items():
    for ep, (loss, dice) in enumerate(
            zip(v['history']['loss'], v['history']['val_dice']), 1):
        history_rows.append({
            'Loss_Function': label,
            'Epoch':         ep,
            'Train_Loss':    round(loss, 6),
            'Val_Dice':      round(dice, 6),
        })
df_history = pd.DataFrame(history_rows)

excel_path = os.path.join(RESULTS_DIR, f'{DOMAIN}_final_results.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_summary.to_excel(writer, sheet_name='Summary',          index=False)
    df_history.to_excel(writer, sheet_name='Training_History', index=False)
print(f'Excel 저장: {excel_path}')
